In [2]:
import pandas as pd
import os
import yt_dlp

In [ ]:
playlist_links = {
    # "NAKO 2019": "https://www.youtube.com/watch?v=t3GQzwxX5uc&list=PL4LQTX97wWcTt0gH0UvN5jtcEpT3Og1Ej", 
    # "NAKO 2020": "https://www.youtube.com/watch?v=KEYXeoyVk50&list=PL4LQTX97wWcR-JeMse88OZ2wOp2hWYZ_u", 
    # "NAKO 2021": "https://www.youtube.com/watch?v=4KNtHYvjcq0&list=PL4LQTX97wWcSJPbsopc3s5RQP3S6dus5Q", 
    # "NAKO 2022": "https://www.youtube.com/watch?v=3HfOtMIw7-M&list=PL4LQTX97wWcS7-pv2GK2f1JK55OVti1m2", 
    # "NAKO 2023": "https://www.youtube.com/watch?v=9GdjwzJub-E&list=PL4LQTX97wWcQT4D2uCL4rGKQ0LvlkjpY_", 
    # "NAKO 2024": "https://www.youtube.com/watch?v=uQEXOc_CPFI&list=PL4LQTX97wWcTfiDhfl3Y905-38gmTmtbJ", 
    # "NAKO 2025": "https://www.youtube.com/watch?v=SGbJgBPaS-0&list=PL4LQTX97wWcTZ5jUPFVnsBS_1xtMSdslJ", 
    # "BATB 2021": "https://www.youtube.com/watch?v=SjUIOcXUQ3A&list=PL4LQTX97wWcTEr8f2358Qaq2u3C_jYR8I", 
    # "BATB 2026": "https://www.youtube.com/watch?v=koIlQQMEq_k&list=PL30GDKFpkkAI8jdIWH_blxMaUqF3OKeLg", 
    # "MKO 2018": "https://www.youtube.com/watch?v=te0-wMFhlqw&list=PL4LQTX97wWcSDNhKBxvVUlKyGfOlLLEzM", 
    # "KWC 2025": "https://www.youtube.com/watch?v=7L2_1If9o1I&list=PL3XnXj6GtXB2rYibLhLzPAjqTq3dp5mcD", 
    # "TKO 2025": "https://youtube.com/playlist?list=PL9JfW0HDDhtuXrVXHpiO4hKYadWvY74QH&si=tXk-yAqqVuH_ew9U", 
    # "TKO 2025-2": "https://youtube.com/playlist?list=PL9JfW0HDDhtuF_oTyBNZTg50YxuKeV210&si=Eh-KSd9yNwqYvQUm", 
}

In [8]:
# Create an output directory for saving the descriptions
output_dir = "youtube_playlist_descriptions"
os.makedirs(output_dir, exist_ok=True)

# Configure yt-dlp settings
ydl_opts = {
    'extract_flat': False,       # Fetch full video data (needed for descriptions), not just meta lists
    'skip_download': True,       # Critical! We only want text data, do NOT download actual video/audio files
    'quiet': True,               # Suppress messy terminal download bar outputs
    'no_warnings': True,
}

print("Starting description extraction...\n")

with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    for name, url in playlist_links.items():
        print(f"Processing: {name}...")
        
        # Safe filename formatting (swapping out spaces)
        safe_filename = name.replace(" ", "_")
        file_path = os.path.join(output_dir, f"{safe_filename}_descriptions.txt")
        
        try:
            # extract_info downloads data structures into a massive Python dictionary
            playlist_data = ydl.extract_info(url, download=False)
            
            # Check if it parsed successfully as a playlist structure
            if 'entries' in playlist_data:
                videos = list(playlist_data['entries'])
                
                with open(file_path, "w", encoding="utf-8") as f:
                    f.write(f"=== PLAYLIST: {name} ===\n")
                    f.write(f"URL: {url}\n")
                    f.write(f"Total Videos Found: {len(videos)}\n")
                    f.write("=" * 40 + "\n\n")
                    
                    for index, video in enumerate(videos, start=1):
                        if not video:  # Skip deleted or unavailable videos in the playlist
                            continue
                            
                        title = video.get('title', 'Unknown Title')
                        video_id = video.get('id', '')
                        video_url = f"https://www.youtube.com/watch?v={video_id}" if video_id else "Unknown URL"
                        description = video.get('description', 'No description available.')
                        
                        # Format and write video text blocks
                        f.write(f"[{index}] TITLE: {title}\n")
                        f.write(f"    URL: {video_url}\n")
                        f.write(f"    DESCRIPTION:\n")
                        # Indent the description slightly for easier reading in the text file
                        indented_desc = "\n".join([f"    | {line}" for line in description.splitlines()])
                        f.write(f"{indented_desc}\n")
                        f.write("-" * 60 + "\n\n")
                        
                print(f" -> Successfully saved {len(videos)} videos to {file_path}")
            else:
                print(f" -> Warning: Link for {name} did not resolve into a standard playlist format.")
                
        except Exception as e:
            print(f" -> Error processing {name}: {e}")

print("\nAll done! Check the 'youtube_playlist_descriptions' folder.")

Starting description extraction...


All done! Check the 'youtube_playlist_descriptions' folder.
